In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
# import seaborn as sns
import re
from numpy import isnan
from collections import defaultdict
from pathlib import Path

In [ ]:
#############################################读取数据###############################################

In [ ]:
###1.读取原始数据

path = "dataset/SPM/Meta_data/"
# R5 = pd.read_csv(path+"AR6_Scenarios_Database_R5_regions_v1.1.csv", engine="c")
# print(R5.head())
R10 = pd.read_csv(path+"AR6_Scenarios_Database_R10_regions_v1.1.csv", engine="c")
print(R10.head())
world = pd.read_csv(path+"AR6_Scenarios_Database_World_v1.1.csv")
print(world.head())

In [ ]:
###2.提取年份，变量名，pairs

year_cols = [str(year) for year in range(2010, 2100+1, 10)]
print(year_cols)
#读取所有变量以及pairs
all_vars = pd.read_csv(path+"Variables48.csv", engine="c")
pairs_metadata = pd.read_csv(path+"pairs_metadata.csv", engine="c")  

#提取all_vars中所有变量的名称
vars = all_vars['Variable'].tolist()
print(len(vars))

#提取pairs_metadata中所有的行的Model Scenario组合
pairs = pairs_metadata[['Model', 'Scenario']].drop_duplicates().reset_index(drop=True)
print(len(pairs))


In [ ]:
#从world、R5、R10中提取1202个pair和50个variable相关的变量
# 从world、R5、R10中提取1202个pair和50个variable相关的变量
# pairs为DataFrame，需用merge方式筛选Model, Scenario组合

world = world.merge(pairs, on=['Model', 'Scenario'], how='inner')
world = world[world['Variable'].isin(vars+["Emissions|CO2|Waste"])]

R10 = R10.merge(pairs, on=['Model', 'Scenario'], how='inner')
R10 = R10[R10['Variable'].isin(vars+["Emissions|CO2|Waste"])]
print(len(vars+["Emissions|CO2|Waste"]))
print(len(R10[['Model', 'Scenario']]))

In [ ]:
#########################################补全waste-other-ge##################################################

In [ ]:

def fill_missing_zero(df, variable, group_cols, year_cols, verbose=True):
    """
    对指定 variable，补全所有缺失的 (group_cols) 组合，所有年份补0。
    """
    group = group_cols[0:3]
    
    all_keys = set(tuple(row) for row in df[group].drop_duplicates().values)
    existing_keys = set(tuple(row) for row in df[df['Variable'] == variable][group].drop_duplicates().values)
    print(len(group),len(all_keys),len(existing_keys))
    missing_keys = all_keys - existing_keys
    rows = []
    for key in missing_keys:
        row = {
            group_cols[0]: key[0],
            group_cols[1]: key[1],
            group_cols[2]: key[2],
            # group_cols[3]: key[3],
            'Variable': variable,
            **{year: 0 for year in year_cols},
        }
        rows.append(row)
    if rows:
        new_df = pd.DataFrame(rows)
        df = pd.concat([df, new_df], ignore_index=True)
        if verbose:
            print(f"已补充 {len(new_df)} 条 {variable} 数据（补0）。")
    else:
        if verbose:
            print(f"没有需要补充的 {variable} 组合。")
    return df



In [ ]:
def fill_emission_co2_other_by_constraint(df, year_cols, verbose=True):
    """
    1. 将Waste数据合并到Other中（相同Model/Scenario/Region/Unit下）
    2. 基于约束条件 CO2 = AFOLU + Process + Other 补全缺失的Other
    """
    # 定义常量
    co2_var = "Emissions|CO2"
    afolu_var = "Emissions|CO2|AFOLU"
    process_var = "Emissions|CO2|Energy and Industrial Processes"
    other_var = "Emissions|CO2|Other"
    waste_var = "Emissions|CO2|Waste"
    key_cols = ["Model", "Scenario", "Region", "Unit"]
    
    # 复制数据框
    df = df.copy()
    
    # 1. 处理Waste数据，合并到Other
    waste_rows = df[df['Variable'] == waste_var]


    # 处理每一行waste数据
    for _, waste_row in waste_rows.iterrows():
        # 构建过滤条件：匹配相同Model/Scenario/Region/Unit的Other行
        mask = df['Variable'] == other_var
        for col in key_cols:
            mask = mask & (df[col] == waste_row[col])
        
        if mask.any():  # 找到对应的Other行
            # 更新Other行，将Waste值加到Other上
            other_idx = df[mask].index
            for year in year_cols:
                if pd.notna(waste_row[year]):
                    if pd.isna(df.loc[other_idx, year]):
                        df.loc[other_idx, year] = waste_row[year]
                    else:
                        df.loc[other_idx, year] += waste_row[year]
        else:  # 没找到对应的Other行，创建新的Other行
            new_row = waste_row.copy()
            new_row['Variable'] = other_var
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    
    # 删除所有Waste行
    df = df[df['Variable'] != waste_var]
    
    if verbose:
        print(f"已将 {len(waste_rows)} 条Waste数据合并到Other中")
    
    # 2. 补全缺失的Other
    # 提取所有变量数据
    co2_df = df[df['Variable'] == co2_var].set_index(key_cols)
    afolu_df = df[df['Variable'] == afolu_var].set_index(key_cols)
    process_df = df[df['Variable'] == process_var].set_index(key_cols)
    other_df = df[df['Variable'] == other_var].set_index(key_cols)
    
    # 找出需要补全的键
    keys_all = set(co2_df.index) & set(afolu_df.index) & set(process_df.index)
    keys_missing_other = keys_all - set(other_df.index)
    
    # 补全缺失的Other
    imputed_rows = []
    for key in keys_missing_other:
        co2 = co2_df.loc[key][year_cols]
        afolu = afolu_df.loc[key][year_cols]
        process = process_df.loc[key][year_cols]
        
        # 计算Other = CO2 - AFOLU - Process
        other = co2 - afolu - process
        
        # 只在数据完整时填充
        valid_mask = ~(co2.isna() | afolu.isna() | process.isna())
        values = {year: (other[year] if valid_mask[year] else float('nan')) 
                 for year in year_cols}
        
        new_row = {
            'Model': key[0],
            'Scenario': key[1],
            'Region': key[2],
            'Unit': key[3],
            'Variable': other_var,
            **values,
        }
        imputed_rows.append(new_row)
    
    # 添加补全的记录
    if imputed_rows:
        df = pd.concat([df, pd.DataFrame(imputed_rows)], ignore_index=True)
        if verbose:
            print(f"已补充 {len(imputed_rows)} 条 {other_var} 数据（由CO2=Process+AFOLU+Other约束补全）")
    elif verbose:
        print(f"没有需要补充的 {other_var} 数据")
    
    return df

In [ ]:

imputing_world = fill_emission_co2_other_by_constraint(world, year_cols)

imputing_R10_list = []
for region, group in R10.groupby('Region'):
    # print(year_cols)
    # print(group.head())
    result = fill_emission_co2_other_by_constraint(group, year_cols)
    imputing_R10_list.append(result)
imputing_R10 = pd.concat(imputing_R10_list, ignore_index=True)

In [ ]:
#对Primary Energy|Geothermal和Secondary Energy|Electricity|Geothermal进行补0操作


# 直接对缺失的 Secondary Energy|Electricity|Geothermal 的pair进行补0，不需要任何约束，与Total无关
group_cols = ['Model', 'Scenario', 'Region', 'Unit']
imputing_world = fill_missing_zero(imputing_world, "Secondary Energy|Electricity|Geothermal", group_cols, year_cols)

imputing_R10_list = []
for region, group in imputing_R10.groupby('Region'):
    result = fill_missing_zero(group, "Secondary Energy|Electricity|Geothermal", group_cols, year_cols)
    imputing_R10_list.append(result)
imputing_R10 = pd.concat(imputing_R10_list, ignore_index=True)

imputing_world = fill_missing_zero(imputing_world, "Primary Energy|Geothermal", group_cols, year_cols)


imputing_R10_list = []
for region, group in imputing_R10.groupby('Region'):
    result = fill_missing_zero(group, "Primary Energy|Geothermal", group_cols, year_cols)
    imputing_R10_list.append(result)
imputing_R10 = pd.concat(imputing_R10_list, ignore_index=True)




In [ ]:
#计算imputing_world中，每个pair中变量的缺失情况
# 统计 imputing_world 中每个 pair（Model, Scenario）下各变量的缺失情况，生成缺失矩阵
def compute_pair_variable_missing_matrix(df, year_cols, vars, pair_cols=["Model", "Scenario"], variable_col="Variable", require_year=None):
    """
    统计每个 pair 下每个变量的缺失情况。
    require_year: 若指定，则该年份有数据才算存在，否则只要有任意年份非空即算存在。
    返回 DataFrame: 行为pair，列为变量，值为True(缺失)/False(不缺失)
    """
    # 只保留关心的变量
    df = df[df[variable_col].isin(vars)]
    # 统计每个pair-variable是否有数据
    if require_year is not None:
        # 只看指定年份
        pivot = df.pivot_table(index=pair_cols, columns=variable_col, values=require_year, aggfunc=lambda x: x.notna().any())
    else:
        # 只要有任意年份非空即算存在
        exist = df.melt(id_vars=pair_cols+[variable_col], value_vars=year_cols, value_name="val")
        exist["has_data"] = exist["val"].notna()
        pivot = exist.groupby(pair_cols+[variable_col])["has_data"].any().unstack(variable_col)
    # 缺失为True，存在为False
    missing = ~pivot.fillna(False)
    return missing

missing_matrix = compute_pair_variable_missing_matrix(imputing_world, year_cols, vars, require_year="2020")
missing_matrix.head()
missing_matrix.to_csv("world_pair_variable_missing_matrix.xlsx")
print("缺失矩阵已保存为 world_pair_variable_missing_matrix.xlsx")

In [ ]:
# 统计每个 pair 的缺失率（有多少变量缺失）
pair_missing_rate = missing_matrix.sum(axis=1) / missing_matrix.shape[1]
pair_missing_rate = pair_missing_rate.reset_index().rename(columns={0: "missing_rate"})
pair_missing_rate.to_csv("world_pair_missing_rate.csv", index=False)
print("每个pair的变量缺失率已保存为 world_pair_missing_rate.csv")

# 统计每个变量的缺失率（有多少pair缺失该变量）
variable_missing_rate = missing_matrix.sum(axis=0) / missing_matrix.shape[0]
variable_missing_rate = variable_missing_rate.reset_index()
variable_missing_rate.columns = ["Variable", "missing_rate"]
variable_missing_rate.to_csv("world_variable_missing_rate.csv", index=False)
print("每个变量的pair缺失率已保存为 world_variable_missing_rate.csv")

In [ ]:
#整理新的pairs，缺失比例大于40%的删掉，并存储新的pairs
filtered_pairs = pair_missing_rate[pair_missing_rate['missing_rate'] <= 0.41][['Model', 'Scenario']]
filtered_pairs = filtered_pairs.drop_duplicates().reset_index(drop=True)
filtered_pairs.to_csv("full_coverage_model_scenario_pairs.csv", index=False)
print(f"筛选后剩余 {len(filtered_pairs)} 个pair，已保存为 full_coverage_model_scenario_pairs.csv")

In [ ]:
#提取imputing_R10、imputing_R5中对应的filtered_pairs和50个vars

print(len(vars))
# filter_R5 = imputing_R5.merge(filtered_pairs, on=['Model', 'Scenario'], how='inner')
# filter_R5 = filter_R5[filter_R5['Variable'].isin(vars)]

filter_R10 = imputing_R10.merge(filtered_pairs, on=['Model', 'Scenario'], how='inner')
filter_R10 = filter_R10[filter_R10['Variable'].isin(vars)]

In [ ]:
# 统计 filter_R10 中每个 pair（Model, Scenario）下各变量的缺失情况，保证所有 filtered_pairs 都统计
def compute_pair_variable_missing_matrix(df, year_cols, vars, all_pairs, pair_cols=["Model", "Scenario"], variable_col="Variable", require_year=None):
    """
    统计每个 pair 下每个变量的缺失情况。
    require_year: 若指定，则该年份有数据才算存在，否则只要有任意年份非空即算存在。
    all_pairs: DataFrame，包含所有需要统计的pair（如filtered_pairs），保证缺失pair也统计。
    返回 DataFrame: 行为pair，列为变量，值为True(缺失)/False(不缺失)
    """
    df = df[df[variable_col].isin(vars)]
    if require_year is not None:
        pivot = df.pivot_table(index=pair_cols, columns=variable_col, values=require_year, aggfunc=lambda x: x.notna().any())
    else:
        exist = df.melt(id_vars=pair_cols+[variable_col], value_vars=year_cols, value_name="val")
        exist["has_data"] = exist["val"].notna()
        pivot = exist.groupby(pair_cols+[variable_col])["has_data"].any().unstack(variable_col)
    # 保证所有pair都在index里
    pivot = pivot.reindex(all_pairs.set_index(pair_cols).index)
    missing = ~pivot.fillna(False)
    return missing



In [ ]:
# 分区域统计R10每个region下pair变量缺失率，并导出csv
region_names = filter_R10['Region'].unique()
for region in region_names:
    region_df = filter_R10[filter_R10['Region'] == region]
    # 只取该region下的pair
    region_pairs = filtered_pairs.copy()
    region_pairs['Region'] = region
    # 统计缺失矩阵
    region_missing_matrix = compute_pair_variable_missing_matrix(
        region_df, year_cols, vars, region_pairs, pair_cols=["Model", "Scenario", "Region"], require_year="2020"
    )
    # 缺失率
    region_pair_missing_rate = region_missing_matrix.sum(axis=1) / region_missing_matrix.shape[1]
    region_pair_missing_rate = region_pair_missing_rate.fillna(1.0)
    region_pair_missing_rate = region_pair_missing_rate.reset_index().rename(columns={0: "missing_rate"})
    # 导出
    region_pair_missing_rate.to_csv(f"R10_region_{region}_pair_missing_rate.csv", index=False)
    print(f"Region {region} 的pair变量缺失率已保存为 R10_region_{region}_pair_missing_rate.csv")

In [ ]:
# 合并 world 和 R10（只保留 filtered_pairs 和 48 个变量），并保存到文件



merged = pd.concat([imputing_world, filter_R10], ignore_index=True)
merged = merged[merged['Variable'].isin(vars)]
merged = merged.merge(filtered_pairs, on=['Model', 'Scenario'], how='inner')
if 'category' in pairs_metadata.columns:
    merged = merged.merge(pairs_metadata[['Model', 'Scenario', 'category']].drop_duplicates(), on=['Model', 'Scenario'], how='left')
print(len(merged))
merged.to_csv("dataset/SPM/Meta_data/AR6_merged_world_r10_filtered.csv", index=False)
print(f"合并后的 world 和 R10（只保留 filtered_pairs 和 48 个变量）已保存为 merged_world_r10_filtered.csv，共 {len(merged)} 条数据。")

In [ ]:
# ===== 第一步：记录路径，读取变量表 =====
path = "dataset/SPM/Meta_data/"
all_vars = pd.read_csv(path+"Variables48.csv", engine="c")
all_variables = all_vars['Variable'].tolist()
print(len(all_variables))


In [ ]:
path = "dataset/SPM/Meta_data/"
time_columns = ['2010', '2020', '2030', '2040', '2050', '2060', '2070', '2080', '2090', '2100']
# ===== 第二步读取完整数据表 =====
data_name = "AR6_merged_world_r10_filtered.csv"
df = pd.read_csv(path+data_name)

print(len(df),df.head())

In [ ]:
def region_missing_rate(df, total_pairs, total_vars):
    # 统计每个Region的实际条目数
    region_counts = df.groupby('Region').size().reset_index(name='actual_count')
    region_counts['expected_count'] = total_pairs * total_vars
    region_counts['missing_rate'] = 1 - region_counts['actual_count'] / region_counts['expected_count']
    print(region_counts[['Region', 'actual_count', 'missing_rate']])
    return region_counts[['Region', 'actual_count', 'missing_rate']]

region_missing_rate(df,1188,48)


In [ ]:
#===== 第三步：找到这些pairs的categories =====
pairs_file_name = "pairs_metadata.csv"
pairs_meta = pd.read_csv(path+pairs_file_name)
merged = df.merge(
    pairs_meta[['Model', 'Scenario', 'Category']],
    on=['Model', 'Scenario'],
    how='left'
)


# 将Category列调整为第6列
if 'Category' in merged.columns:
    cols = list(merged.columns)
    cat_idx = cols.index('Category')
    if cat_idx != 5:
        cols.insert(5, cols.pop(cat_idx))
        merged = merged[cols]

print(merged.head())

In [ ]:
#===== 第四步：做一些基本的检查 =====

In [ ]:
unique_models = merged['Model'].dropna().unique()
print(f"\nModel 列中共有 {len(unique_models)} 个不同的模型：")
print(unique_models)
unique_Scenario = merged['Scenario'].dropna().unique()
print(f"\nScenario 列中共有 {len(unique_Scenario)} 个不同的情景：")

# pd.Series(unique_models).to_csv("models.csv", index=False, header=False)
# for model in sorted(unique_models):
#     print(f"- {model}")

unique_Variable = merged['Variable'].dropna().unique()
print(f"\nVariable 列中共有 {len(unique_Variable)} 个不同的变量：")
# for model in sorted(unique_models):
#     print(f"- {model}")

unique_regions = merged['Region'].dropna().unique()
print(f"\nRegion 列中共有 {len(unique_regions)} 个不同的地区：")
# for region in sorted(unique_regions):
#     print(f"- {region}")

unique_Category = merged['Category'].value_counts()
print(f"\nCategory 列中共有 {len(unique_Category)} 个不同的类别：")
# for region, count in region_counts.items():
#     print(f"{region}: {count}")

In [ ]:
# ==== 第五步：删除 Region 为 'R10ROWO' 的行 ====
merged = merged[merged["Region"] != "R10ROWO"]



# ==== 第六步：简单的描述性统计 ====

# 1. 各模型组数量
model_counts = merged["Model"].value_counts()

# 2. 每个变量的数量
variable_counts = merged["Variable"].value_counts()

# 3. 每个地区的数量
region_counts = merged["Region"].value_counts()

#==== 输出结果 ====
print("模型组分布：")
print(model_counts)

print("变量分布（共保留49个变量）：")
print(variable_counts)

print("地区分布（已去除R5ROWO）：")
print(region_counts)






In [ ]:
# ==== 第七步：对时间维度进行插补，并保留有效列，四舍五入 ====


# 时间维度的插补逻辑：
# 如果某一时间列为 NaN，尝试用该行中前一个和后一个非 NaN 值的平均来填补；
# 如果前或后一个值缺失，则退化为单边填充；
# 如果全是 NaN，不填补（保留为 NaN）；
def impute_nan_with_row_neighbors(df, time_columns):
    count = 0
    count1 = 0
    count2 = 0
    """
    用同一行中前后值的均值填补时间序列中的NaN值
    :param df: 原始DataFrame
    :param time_columns: 时间列列表
    :return: 填补后的DataFrame
    """
    time_data = df[time_columns].values.astype(float)

    for i in range(time_data.shape[0]):  # 遍历每一行
        for j in range(time_data.shape[1]):
            if np.isnan(time_data[i, j]):
                prev_val = next_val = None

                # 查找前一个非NaN
                for k in range(j - 1, -1, -1):
                    if not np.isnan(time_data[i, k]):
                        prev_val = time_data[i, k]
                        break

                # 查找后一个非NaN
                for k in range(j + 1, time_data.shape[1]):
                    if not np.isnan(time_data[i, k]):
                        next_val = time_data[i, k]
                        break

                # 插补逻辑
                if prev_val is not None and next_val is not None:
                    time_data[i, j] = (prev_val + next_val) / 2
                    count += 1
                elif prev_val is not None:
                    time_data[i, j] = prev_val
                    count1 += 1
                elif next_val is not None:
                    time_data[i, j] = next_val
                    count2 += 1
                # 否则保留为 NaN
    print(f"双侧填充的次数为：{count}，单边填充前值的次数: {count1}, 后值的次数: {count2}")
    df[time_columns] = time_data
    return df

# 应用插补（假设时间列从第10列开始）
time_columns = ['2010', '2020', '2030', '2040', '2050', '2060', '2070', '2080', '2090', '2100']  # 索引9对应Excel第10列
print(time_columns)
# print(merged[time_columns].head())
print(merged.head())
merged_uslcol = impute_nan_with_row_neighbors(merged, time_columns)



# #仅仅保留有效列，并对time_columns进行四舍五入

useful_col = merged_uslcol.columns[[0,1,2,3,5]].append(merged_uslcol.columns[21::10])
# useful_col = merged_uslcol.columns[[0,1,2,3,5]].append(merged_uslcol.columns[6:16])

print(useful_col)

merged_uslcol = merged_uslcol[useful_col].round(4)
# merged_uslcol = merged_uslcol.round(4)


print(merged_uslcol.head())


In [ ]:

# #修改model列，以【AIM COFFEE GCAM GEM IMAGE MESSAGE POLES REMIND TIAM WITCH 】为前缀的model名合并为同一个类型
# model_prefixes = ["AIM", "COFFEE", "GCAM", "GEM", "IMAGE", "MESSAGE", "POLES", "REMIND", "TIAM", "WITCH"]
# def merge_model_prefixes(model_name):
#     for prefix in model_prefixes:
#         if model_name.startswith(prefix):
#             return prefix
#     return model_name
# merged_uslcol['Model'] = merged_uslcol['Model'].apply(merge_model_prefixes)

os.makedirs('变量编码对应表', exist_ok=True)

dim_encoders = []

for col_idx, col_name in enumerate(['Model', 'Category', 'Region', 'Variable']):
    unique_vals = merged_uslcol[col_name].unique()
    encoder = {v: i for i, v in enumerate(unique_vals)}
    dim_encoders.append(encoder)
    
    # 保存编码表
    pd.DataFrame({'Value': list(encoder.keys()), 'Code': list(encoder.values())}) \
      .to_csv(f'dataset/SPM/变量编码对应表/dim_{col_name}_encoding.csv', index=False)

print('编码完成，共生成', len(dim_encoders), '个维度的编码表')




In [ ]:
#交换第二列和第五列 ['Model', 'Scenario', 'Region', 'Variable', 'Category', '2010', '2020',
merged_uslcol = merged_uslcol[[merged_uslcol.columns[0], merged_uslcol.columns[4], merged_uslcol.columns[2], merged_uslcol.columns[3], merged_uslcol.columns[1]] + list(merged_uslcol.columns[5:])]
print(merged_uslcol.head())



In [ ]:
from tqdm import tqdm
import numpy as np

def build_spm_fixed_grid(merged_uslcol, time_columns, dim_encoders,
                         group_cols=('Model','Scenario')):
    enc_model, enc_category, enc_region, enc_variable = dim_encoders

    # 1) 用编码顺序确定全局 Region / Variable 顺序
    region_order = [name for name, _ in sorted(enc_region.items(),   key=lambda kv: kv[1])]
    var_order    = [name for name, _ in sorted(enc_variable.items(), key=lambda kv: kv[1])]

    R, V = len(region_order), len(var_order)
    T, G = len(time_columns), R * V

    # 2) (Region, Variable) -> 固定下标
    rv2k = {(r, v): (ri * V + vi)
            for ri, r in enumerate(region_order)
            for vi, v in enumerate(var_order)}

    # 3) 预先生成 group 轴上的编码并在时间维广播用
    reg_codes = np.repeat([enc_region[r]   for r in region_order], repeats=V).astype(np.int32)  # (G,)
    var_codes = np.tile  ([enc_variable[v] for v in var_order],   reps=R).astype(np.int32)      # (G,)

    # 4) 分组（每个 (Model,Scenario) 是一个样本）
    grouped = merged_uslcol.groupby(list(group_cols), sort=False)
    N = len(grouped)

    padded = np.zeros((N, G, T, 5), dtype=np.float32)
    mask   = np.zeros((N, G),       dtype=np.uint8)

    # 进度条
    for i, (key, g) in enumerate(tqdm(grouped, total=N, desc="Building fixed grid", ncols=100)):
        g = g.reset_index(drop=True)

        model_name    = g.loc[0, 'Model']
        category_name = g.loc[0, 'Category'] if 'Category' in g.columns else None
        m_code = enc_model.get(model_name, 0)
        c_code = enc_category.get(category_name, 0)

        # 一次性写满 4 个编码列
        padded[i, :, :, 0] = m_code
        padded[i, :, :, 1] = c_code
        padded[i, :, :, 2] = reg_codes[:, None]  # (G,1) 广播到 (G,T)
        padded[i, :, :, 3] = var_codes[:, None]

        # 填值 & mask
        seen = set()
        for _, row in g.iterrows():
            r, v = row['Region'], row['Variable']
            k = rv2k.get((r, v))
            if k is not None and k not in seen:
                padded[i, k, :, 4] = row[time_columns].to_numpy(dtype=np.float32)
                mask[i, k] = 1
                seen.add(k)

    return padded, mask, region_order, var_order

# 用法
padded_array, mask_array, region_list, var_list = build_spm_fixed_grid(merged_uslcol, time_columns, dim_encoders)
print(padded_array.shape, mask_array.shape)


In [ ]:
data = padded_array[padded_array[:,:,:,3] == 35]

print(data.mean())

In [ ]:
np.savez("dataset/SPM/spm.npz", padded=padded_array, mask=mask_array)


In [ ]:
# 统计padded_array中，每个（model，cluster）下包含多少个组

model_cluster = padded_array[:, :, 0, :2]
group_counts = defaultdict(int)
for i in range(model_cluster.shape[0]):
    key = tuple(model_cluster[i,0,...])  # 将数组转换为列表，然后转换为元组
    print(key)
    group_counts[key] += 1


In [ ]:
# 将计数转换为Series
counts_series = pd.Series(group_counts)

# 获取详细统计信息
stats = counts_series.describe()

print("详细统计信息:")
print(stats)

In [ ]:
print(len(filter_R10))
print(len(imputing_world))

In [ ]:
# 检查 filter_R10 中各个区域的 pair 数量和变量数量
region_stats = []
for region in merged['Region'].unique():
    region_df = merged[merged['Region'] == region]
    n_pairs = region_df[['Model', 'Scenario']].drop_duplicates().shape[0]
    n_vars = region_df['Variable'].nunique()
    region_stats.append({'Region': region, 'Pair数': n_pairs, '变量数': n_vars})
region_stats_df = pd.DataFrame(region_stats)
print(region_stats_df)

In [ ]:
# 查看 merged 各个 region 的数据条数
region_counts = merged['Region'].value_counts().reset_index()
region_counts.columns = ['Region', 'Count']
print(region_counts)